# Comparison

This notebook loads saved SNN and Gaussian results. It never reruns either simulation.

## 1. Select the SNN and approximation runs

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "01_Original_Colab").is_dir()), Path.cwd())
SNN_RUN_ID = "holy kale-faithful-snn"
APPROX_RUN_ID = "holy kale-gaussian-15hz"
SNN_RESULTS = ROOT / "02_Brunel_SNN" / "results"
APPROX_RESULTS = ROOT / "03_Gaussian_Approximation" / "results"
FIGURES = ROOT / "04_Comparison" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)


## 2. Check model parameters and analysis conventions

In [ ]:
def load_run(directory, run_id):
    data_path, meta_path = directory / f"{run_id}.npz", directory / f"{run_id}.json"
    if not data_path.exists() or not meta_path.exists():
        raise FileNotFoundError(f"Missing saved run {run_id!r}; run the source notebook first.")
    return np.load(data_path), json.loads(meta_path.read_text())

snn, snn_meta = load_run(SNN_RESULTS, SNN_RUN_ID)
approx, approx_meta = load_run(APPROX_RESULTS, APPROX_RUN_ID)
print("SNN duration (s):", snn_meta["parameters"]["duration_s"])
print("Approximation duration (s):", approx_meta["measurement_duration_s"])
print("SNN convention:", snn_meta["analysis"]["spectrum"])
print("Approximation convention:", approx_meta["analysis"]["spectrum"])


## 3. Compare firing rates

In [ ]:
snn_rate = float(snn["mean_rate_hz"]); approx_rate = float(approx["rates_hz"][-1])
print(f"SNN: {snn_rate:.3f} Hz; approximation final generation: {approx_rate:.3f} Hz")
print(f"Absolute error: {abs(snn_rate-approx_rate):.3f} Hz")
fig, ax = plt.subplots(figsize=(6, 4)); ax.bar(["SNN", "Approximation"], [snn_rate, approx_rate], color=["deeppink", "royalblue"])
ax.set_ylabel("Mean firing rate (Hz)"); ax.set_title("Firing-rate comparison"); fig.tight_layout(); fig.savefig(FIGURES / "firing_rate.png", dpi=150); plt.show()


## 4. Compare ISI CV

The SNN CV averages variability across network neurons over 3 seconds. The approximation CV describes one neuron over a 500-second temporal record. These are not identical estimands; the comparison assumes stationarity and representative neurons.

In [ ]:
snn_cv = float(snn["mean_cv"]); approx_cv = float(approx["cvs"][-1])
print(f"SNN mean CV: {snn_cv:.3f}; approximation final CV: {approx_cv:.3f}")
fig, ax = plt.subplots(figsize=(6, 4)); ax.bar(["SNN", "Approximation"], [snn_cv, approx_cv], color=["deeppink", "royalblue"])
ax.set_ylabel("ISI coefficient of variation"); ax.set_title("CV comparison"); fig.tight_layout(); fig.savefig(FIGURES / "cv.png", dpi=150); plt.show()


## 5. Overlay single-neuron spectra

The original SNN spectrum divides the two-sided FFT power by 2, while Welch is one-sided by default. The original measurements also have different durations and bin widths. The first panel preserves those outputs. The corrected panel uses unsmoothed interpolation onto the SNN frequency grid and applies a documented one-sided conversion to the SNN positive frequencies; it is a comparison aid, not a rerun.

In [ ]:
sf, ss = snn["frequency_hz"], snn["spectrum"]; af = approx["frequencies_hz"][-1]; ap = approx["spectra"][-1]
mask_s = (sf > 0) & (sf <= 400); mask_a = (af > 0) & (af <= 400)
fig, ax = plt.subplots(figsize=(10, 4)); ax.semilogy(sf[mask_s], ss[mask_s], color="deeppink", label="SNN original"); ax.semilogy(af[mask_a], ap[mask_a], color="royalblue", label="Approximation original")
ax.set(xlabel="Frequency (Hz)", ylabel="Unsmoothed power / PSD", title="Original spectrum conventions"); ax.legend(); fig.tight_layout(); fig.savefig(FIGURES / "spectra_original.png", dpi=150); plt.show()
common_spectrum = np.interp(sf[mask_s], af[mask_a], ap[mask_a])
snn_corrected = 2.0 * ss[mask_s]
fig, ax = plt.subplots(figsize=(10, 4)); ax.semilogy(sf[mask_s], snn_corrected, color="deeppink", label="SNN corrected: undo /2"); ax.semilogy(sf[mask_s], common_spectrum, color="royalblue", label="Approximation rebinned to SNN grid")
ax.set(xlabel="Frequency (Hz)", ylabel="Unsmoothed common-grid power", title="Corrected comparison (linear interpolation; no smoothing)"); ax.legend(); fig.tight_layout(); fig.savefig(FIGURES / "spectra_corrected.png", dpi=150); plt.show()


## 6. Report errors and limitations

In [ ]:
relative_rate_error = abs(snn_rate - approx_rate) / abs(snn_rate)
relative_cv_error = abs(snn_cv - approx_cv) / abs(snn_cv)
print(f"Rate relative error: {100*relative_rate_error:.3f}%")
print(f"CV relative error: {100*relative_cv_error:.3f}%")
print("Limitations: different durations, estimators, bin widths, and spatial-vs-temporal CV estimands.")


## 7. Save figures and a small summary table

In [ ]:
summary = {"snn_run_id": SNN_RUN_ID, "approximation_run_id": APPROX_RUN_ID, "snn_duration_s": snn_meta["parameters"]["duration_s"], "approximation_duration_s": approx_meta["measurement_duration_s"], "snn_rate_hz": snn_rate, "approximation_rate_hz": approx_rate, "rate_absolute_error_hz": abs(snn_rate-approx_rate), "snn_cv": snn_cv, "approximation_cv": approx_cv, "cv_absolute_error": abs(snn_cv-approx_cv), "corrected_spectrum": "SNN positive-frequency power multiplied by 2; approximation linearly interpolated to SNN grid; no smoothing"}
(ROOT / "04_Comparison" / "results").mkdir(parents=True, exist_ok=True)
(ROOT / "04_Comparison" / "results" / "comparison_summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
